# Methods statistics

Recomputes the numbers that the main Methods prints and that no figure or table carries: reservoir sizes, comparator FASTA sizes, the reconverted PREDICT files, the cross-fit roster, the selected-FASTA unions, the resampling settings of the spectrum-split and antibiotic analyses, and the compute times. Every value is computed from `source_data/` (and, for the antibiotic test only, from a controlled input).

Run from the repository root: `python paper.py build methods`.

In [1]:
from pathlib import Path
import sys
UNIT = Path.cwd()                 # the notebook runs with its own directory as cwd
REPO = UNIT.parents[1]
sys.path[:0] = [str(REPO), str(UNIT)]
SD = REPO / "source_data" / UNIT.name
import pandas as pd
pd.set_option("display.width", 200)
import stats
R = {}                            # every recomputed number, keyed as in NUMBERS.tsv

## Line 215: reservoirs

"The public-sequence reservoir comprised 233,497,702 proteins ... cohort reservoirs containing proteins pooled from all 296 PREDICT or 335 CapScan assembly FASTAs, giving 272,919,212 and 264,925,283 sequences ... The CAMPI reservoir ... giving 375,333,654 sequences" (v28d line 215).

n: one reservoir each; unit: sequence. Descriptive count: the sum of the unique sequences each source added in priority order, from the lake-builder statistics. Checked against the reservoir sizes in `source_data/Fig1/acquisition_sizes_726.tsv`.

In [2]:
res = stats.stat_reservoirs()
print(res)
assert res["fig1_agrees"]
R.update({"233,497,702": res["size"]["public"], "296": res["assembly_fastas"]["PREDICT"], "335": res["assembly_fastas"]["CapScan"],
          "272,919,212": res["size"]["PREDICT_cohort"], "264,925,283": res["size"]["CapScan_cohort"], "375,333,654": res["size"]["CAMPI"]})

{'size': {'CAMPI': 375333654, 'CapScan_cohort': 264925283, 'PREDICT_cohort': 272919212, 'public': 233497702}, 'assembly_fastas': {'PREDICT': 296, 'CapScan': 335}, 'fig1_agrees': True}


## Line 227: CAMPI comparator FASTAs

"The reference and MG FASTAs contained 29,673 and 19,435 records, each including 116 contaminants" (v28d line 227).

n: 29,673 and 19,435 records; unit: sequence. Descriptive count of the FASTAs as frozen for the comparator arms (supplied decoys removed). Contaminants are the records whose accession occurs in both FASTAs; all other MG records are assembly proteins (`k99_` accessions).

In [3]:
cmp_ = stats.stat_comparator_fastas()
print(cmp_)
assert cmp_["mg_non_contaminants_all_k99"] and cmp_["contaminant_sequences_identical"]
assert cmp_["contaminants"]["reference"] == cmp_["contaminants"]["MG"]
R.update({"29,673": cmp_["records"]["reference"], "19,435": cmp_["records"]["MG"], "116": cmp_["contaminants"]["reference"]})

{'records': {'reference': 29673, 'MG': 19435}, 'contaminants': {'reference': 116, 'MG': 116}, 'mg_non_contaminants_all_k99': True, 'contaminant_sequences_identical': True, 'accessions_unique_within_database': True}


## Lines 233 and 303: reconverted PREDICT mzML files

"Five PREDICT mzML files that were truncated on disk were reconverted from the RAW files before all searches reported here" (line 233); "For five PREDICT acquisitions (MicrobPred_13, 44, 90, 115 and 181), the deposited mzML files are the validated reconversions" (line 303).

n: 306 PREDICT public-sequence searches; unit: acquisition. Descriptive: which searched mzML files were reconversions, and the end-of-file check of the original files.

In [4]:
rec = stats.stat_reconversion()
print(rec)
assert rec["reconverted_were_truncated"] and rec["others_complete"] and rec["all_searched_mzml_closed"]
R.update({"Five": rec["n_word"].capitalize(), "five": rec["n_word"], "MicrobPred_13, 44, 90, 115 and 181": rec["text"]})

{'n': 5, 'n_word': 'five', 'labels': ['13', '44', '90', '115', '181'], 'text': 'MicrobPred_13, 44, 90, 115 and 181', 'reconverted_were_truncated': True, 'others_complete': True, 'all_searched_mzml_closed': True, 'n_searches': 306}


## Line 267: spectrum-split bootstrap

"We summarized gains as acquisition medians with 95% intervals from 10,000 participant-level bootstrap resamples" (v28d line 267).

n: 296 PREDICT acquisitions from 97 patients; 282 CapScan acquisitions from 15 participants (56 stool, 226 intestinal fluid). Unit of resampling: patient or participant. Interval: percentile, 10,000 draws, seed 20261003. The recomputation is compared with the published summary (`split_spectra_mg_summary_original.tsv`) at its two decimals.

In [5]:
boot = stats.split_bootstrap()
cmpb = stats.compare_split_bootstrap(boot)
ok = cmpb.median_gain_pct_agrees & cmpb.ci95_low_agrees & cmpb.ci95_high_agrees
print(f"{int(ok.sum())} of {len(ok)} summary rows reproduced (median and both bounds, 2 decimals)")
print(boot[["stratum", "min_length", "comparison", "n_acquisitions", "n_participants", "median_gain_pct", "ci95_low", "ci95_high"]].to_string(index=False))
assert ok.all()
R["10,000"] = stats.SPLIT_BOOT_DRAWS

40 of 40 summary rows reproduced (median and both bounds, 2 decimals)
           stratum  min_length                                comparison  n_acquisitions  n_participants  median_gain_pct   ci95_low  ci95_high
           PREDICT           5                   gain_frozen_fullspectra             296              97        18.754795  16.818540  20.723865
           PREDICT           5                             gain_odd_full             296              97        20.198447  18.702685  22.849609
           PREDICT           5                             gain_odd_self             296              97        12.412184  10.508029  14.768722
           PREDICT           5                         gain_odd_disjoint             296              97        -0.988458  -3.007630   1.108987
           PREDICT           5 disjoint_minus_full_selection_gain_points             296              97       -21.428326 -21.942846 -21.019579
           PREDICT           9                   gain_frozen_fulls

## Line 267: cross-fit roster, singular fits and interpreter

"Where this fit was singular, in three acquisitions, Sage's heuristic score was used ... Pooled rescoring ran in Python 3.10.9 with NumPy 1.26.4. All 578 matched acquisitions were analyzed" (v28d line 267).

n: 578 acquisitions (296 PREDICT, 282 CapScan); unit: acquisition. Counts from the production cross-fit table, repeated on the rerun with 347-acquisition CapScan caches. The interpreter versions were read from the interpreter named in the cross-fit launcher.

In [6]:
cf = stats.stat_crossfit_roster()
env = stats.stat_crossfit_interpreter()
print(cf); print(env)
assert cf["fallback_index_equal"] and cf["n_347"] == cf["n"]
R.update({"three": cf["n_fallback_word"], "578": cf["n"], "3.10.9": env["python"], "1.26.4": env["numpy"]})

{'n': 578, 'n_by_cohort': {'PREDICT': 296, 'CAPSCAN': 282}, 'n_fallback': 3, 'n_fallback_word': 'three', 'fallback': [('CAPSCAN', 'FL'), ('PREDICT', 'FL;FULL'), ('PREDICT', 'FL;FULL')], 'n_347': 578, 'n_fallback_347': 3, 'fallback_347': [('CAPSCAN', 'FL'), ('PREDICT', 'FL;FULL'), ('PREDICT', 'FL;FULL')], 'fallback_index_equal': True}
{'python': '3.10.9', 'numpy': '1.26.4'}


## Line 269: study unions of the selected FASTAs

"Exact sequence deduplication of the selected FASTAs ... produced study unions of 153,267 healthy-stool, 555,987 PREDICT, and 452,255 CapScan proteins" (v28d line 269).

n: 58, 306 and 331 or 347 FASTAs; unit: protein sequence (exact, I/L distinct). The union is the sum of the sequences each FASTA adds. 452,255 is the 331-acquisition CapScan roster; the 347-acquisition roster used for v28f gives the value below.

In [7]:
un = stats.stat_unions()
print(pd.DataFrame(un).T)
R.update({"153,267": un["HSTOOL58"]["unique"], "555,987": un["PREDICT306"]["unique"], "452,255": un["CAPSCAN347"]["unique"]})

            fastas  records  unique
CAPSCAN331     331  2656239  452255
CAPSCAN347     347  2767715  460977
HSTOOL58        58   709676  153267
PREDICT306     306  2363854  555987


## Line 275: antibiotic comparison

"For 42 PREDICT patients observed off and on antibiotics ... Mean changes used 50,000 patient-bootstrap resamples. Two-sided tests used 200,000 sign assignments with plus-one correction and Benjamini–Hochberg adjustment across human-assigned share and protein-group depth" (v28d line 275).

n: 42 patients; unit: patient. Test: Monte Carlo paired sign-flip test of the mean difference, BH across 2 endpoints; interval: percentile patient bootstrap. The per-patient values are a controlled input; without `FASTALAKE_CONTROLLED_DIR` the cell prints the public aggregate table only.

In [8]:
cl = stats.stat_clinical()
print(cl["public"].to_string(index=False))
if cl["recomputed"] is not None:
    print(cl["recomputed"].to_string(index=False))
    pub = cl["public"].set_index("metric")
    for col in ["mean_on_minus_off", "CI95_low", "CI95_high", "p_signflip_MC", "q_two_clinical_endpoints"]:
        assert all(abs(cl["recomputed"].set_index("metric")[col] - pub[col]) <= 1e-9 * abs(pub[col]).clip(lower=1)), col
    print("recomputation equals the public table")
R.update({"42": cl["n_patients"], "50,000": cl["draws"], "200,000": cl["signflips"]})

      metric  n_patients    mean_off     mean_on  mean_on_minus_off     CI95_low   CI95_high                        units  p_signflip_MC  q_two_clinical_endpoints
       Human          42   19.297668   30.430990          11.133322     5.559830   17.156839            percentage points       0.000340                  0.000680
pg_parsimony          42 9319.107143 7105.154762       -2213.952381 -3469.322024 -997.525595 quantified simplified groups       0.001255                  0.001255
      metric  n_patients    mean_off     mean_on  mean_on_minus_off     CI95_low   CI95_high                        units  p_signflip_MC  exceed_count  q_two_clinical_endpoints
       Human          42   19.297668   30.430990          11.133322     5.559830   17.156839            percentage points       0.000340            67                  0.000680
pg_parsimony          42 9319.107143 7105.154762       -2213.952381 -3469.322024 -997.525595 quantified simplified groups       0.001255           250       

## Line 299: compute times and hardware

"CAMPI reservoir construction required 92 min on a 64-thread, 512-GB AMD EPYC node. Pooled extraction took 11 min. Median per-acquisition times for extraction, selection, and Sage/LFQ searching were 0.48, 0.63, and 0.23 min, respectively. AlphaNovo required a median of 28 min per CAMPI acquisition on one NVIDIA L40S GPU" (v28d line 299).

n: 1 reservoir job, 1 pooled extraction job, 15 tasks per acquisition stage, 15 inference tasks; unit: Slurm job or task. Descriptive (wall time from Slurm accounting; 64 is the lake-builder thread setting; node memory from `scontrol`).

In [9]:
cpu = stats.stat_compute()
print(cpu)
R.update({"92": cpu["lake_minutes"], "64": cpu["builder_threads"], "512": cpu["node_memory_GB"], "11": cpu["pooled_extraction_minutes"],
          "0.48": cpu["stage_medians"]["extraction"], "0.63": cpu["stage_medians"]["selection"], "0.23": cpu["stage_medians"]["search"],
          "28": cpu["inference_median_minutes"]})

{'lake_minutes': 92.36666666666666, 'lake_jobid': '5895248', 'lake_node': 'hpcl9202', 'builder_threads': 64, 'node_memory_GB': 512.0, 'node_features': 'genoa,mem512G', 'node_logical_cpus': 128, 'pooled_extraction_minutes': 11.066666666666666, 'stage_medians': {'extraction': 0.48333333333333334, 'selection': 0.6333333333333333, 'search': 0.23333333333333334}, 'stage_n': {'extraction': 15, 'selection': 15, 'search': 15}, 'stage_medians_summary_table': {'extraction': 0.4833333333333333, 'selection': 0.6333333333333333, 'search': 0.2333333333333333}, 'inference_median_minutes': 28.266666666666666, 'inference_n': 15, 'inference_gres': ['gpu:l40s:4(S:0)']}


## Checks

This unit has no figure panels, so there are no pixel checks. The cell prints every recomputed number in `NUMBERS.tsv` order and asserts that it equals the `recomputed_value` column.

In [10]:
import csv
rows = list(csv.DictReader(open(UNIT / "NUMBERS.tsv", encoding="utf-8"), delimiter="\t"))
for r in rows:
    v = R[r["printed_value"]]
    shown = repr(v) if isinstance(v, float) else str(v)
    assert shown == r["recomputed_value"], (r["printed_value"], shown, r["recomputed_value"])
    print(r["v28d_line"], r["printed_value"], shown, r["status"], sep="\t")
print("pixel checks: none (no quantitative panels in this unit)")

215	233,497,702	233497702	MATCH
215	296	296	MATCH
215	335	335	MATCH
215	272,919,212	272919212	MATCH
215	264,925,283	264925283	MATCH
215	375,333,654	375333654	MATCH
227	29,673	29673	MATCH
227	19,435	19435	MATCH
227	116	116	MATCH
233	Five	Five	MATCH
267	10,000	10000	MATCH
267	three	three	MATCH
267	3.10.9	3.10.9	MATCH
267	1.26.4	1.26.4	MATCH
267	578	578	MATCH
269	153,267	153267	MATCH
269	555,987	555987	MATCH
269	452,255	460977	v28f_change:460,977
275	42	42	MATCH
275	50,000	50000	MATCH
275	200,000	200000	MATCH
299	92	92.36666666666666	MATCH
299	64	64	MATCH
299	512	512.0	MATCH
299	11	11.066666666666666	MATCH
299	0.48	0.48333333333333334	MATCH
299	0.63	0.6333333333333333	MATCH
299	0.23	0.23333333333333334	MATCH
299	28	28.266666666666666	MATCH
303	five	five	MATCH
303	MicrobPred_13, 44, 90, 115 and 181	MicrobPred_13, 44, 90, 115 and 181	MATCH
pixel checks: none (no quantitative panels in this unit)
